# Laboratorio 15: Consulta y Comparación de Bases Vectoriales

Este notebook permite:
- Crear la base de datos vectorial `Tecsup` y `Tecsup_Custom` si están vacías.
- Consultar una pregunta y comparar los resultados obtenidos por dos motores de embeddings:
  - `DefaultEmbeddingFunction`
  - `SentenceTransformer`

In [1]:
import chromadb
import logging
from chromadb.utils import embedding_functions
from src.embeddings.embedding_engine import get_sentence_transformer
from src.vector_db.crear_tecsup_db import cargar_tecsup_default
from src.vector_db.crear_tecsup_custom_db import cargar_tecsup_custom

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

In [2]:
# Cliente a la base de datos persistente
client = chromadb.PersistentClient(path="data/processed/chroma_tecsup")

2025-06-29 10:18:36,672 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2025-06-29 10:18:36,773 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [3]:
# Crear las base de datos si están vacías
cargar_tecsup_default()
cargar_tecsup_custom()

2025-06-29 10:18:39,210 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2025-06-29 10:18:39,226 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2025-06-29 10:18:39,237 - INFO - Colección 'Tecsup' eliminada antes de recrearla.
2025-06-29 10:18:39,238 - ERROR - Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
2025-06-29 10:18:41,022 - ERROR - Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
2025-06-29 10:18:41,101 - INFO - Documentos insertados en la colección 'Tecsup': 20
2025-06-29 10:18:41,107 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2025-06-29 10:18:41,115 - INFO - Colección 'Tecsup_Custom' eliminada antes de recrearla.
C:\Proyectos_Pycharm\Laboratorio_1

In [4]:
# Prompt
prompt = "¿Quién estudia ciencia de datos o le interesa la inteligencia artificial?"
logging.info(f"Consulta definida: {prompt}")

2025-06-29 10:18:59,245 - INFO - Consulta definida: ¿Quién estudia ciencia de datos o le interesa la inteligencia artificial?


In [5]:
# Consulta con DefaultEmbeddingFunction

# Recargar el cliente (para asegurar consistencia)
client = chromadb.PersistentClient(path="data/processed/chroma_tecsup")

# Consulta con DefaultEmbeddingFunction
default_func = embedding_functions.DefaultEmbeddingFunction()
collection_default = client.get_or_create_collection(name="Tecsup", embedding_function=default_func)

# Confirmar que hay documentos
count_default = collection_default.count()
logging.info(f"Número total de documentos en 'Tecsup': {count_default}")

if count_default > 0:
    res_default = collection_default.query(query_texts=[prompt], n_results=3)
    logging.info("Resultados más similares:")
    for i, doc in enumerate(res_default["documents"][0]):
        distancia = res_default["distances"][0][i]
        logging.info(f"[Default] Documento {i+1} (Distancia: {distancia:.4f})")
        for linea in doc.split('. '):
            logging.info(f"   {linea.strip()}")
else:
    logging.warning("La colección 'Tecsup' no contiene documentos.")


2025-06-29 10:19:06,150 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2025-06-29 10:19:06,151 - ERROR - Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
2025-06-29 10:19:06,157 - INFO - Número total de documentos en 'Tecsup': 0
2025-06-29 10:19:06,158 - WARNING - La colección 'Tecsup' no contiene documentos.


In [ ]:
# Consulta con SentenceTransformer
custom_func = get_sentence_transformer()
collection_custom = client.get_or_create_collection(name="Tecsup_Custom", embedding_function=custom_func)
count_custom = collection_custom.count()
logging.info(f"Número total de documentos en 'Tecsup_Custom': {count_custom}")

if count_custom > 0:
    res_custom = collection_custom.query(query_texts=[prompt], n_results=3)
    for i, doc in enumerate(res_custom["documents"][0]):
        logging.info(f"[Custom] Documento {i+1} - Distancia: {res_custom['distances'][0][i]:.4f}")
        logging.info(doc)
else:
    logging.warning("La colección 'Tecsup_Custom' no contiene documentos.")